# Por que cada alerta disparou

Cada alerta da API vem com os três atributos que mais pesaram nele
([ADR 0013](../docs/adr/0013-explicacao-dos-alertas.md)):

- num `attack`, a contribuição de cada atributo para a família prevista, calculada pelo próprio
  XGBoost (`pred_contribs`, o SHAP exato das árvores);
- num `suspicious`, o erro de reconstrução de cada atributo no autoencoder: o que mais fugiu do
  tráfego normal que ele aprendeu.

Este notebook passa o teste do P2 (quinta e sexta, só famílias que os modelos nunca viram) pelo
mesmo par de modelos do painel, `xgb_unseen` e `ae_unseen`, e pergunta: **as explicações mostram o
que os modelos de fato usam?** Em especial, o atalho da janela TCP do notebook 02 (seção 9), que
identifica a máquina atacante em vez do ataque, aparece alerta por alerta?

**Como rodar:** com o conjunto original processado (`nids data split`) e os modelos em `models/`
(treinados ou baixados da release). Leva alguns minutos.

In [1]:
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from nids import schema
from nids.serving.predictor import load_predictor

# Os atributos que dependem do sistema operacional da máquina (notebook 02, seção 9)
TCP_WINDOW = {"init_win_bytes_forward", "init_win_bytes_backward", "min_seg_size_forward"}
pd.set_option("display.max_colwidth", None)

test = pd.read_parquet("../data/processed/unseen/test.parquet")
predictor = load_predictor(Path("../models"), "xgb_unseen", "ae_unseen")
print(f"{len(test):,} fluxos no teste do P2".replace(",", "."))
print("classes do XGBoost:", predictor.classifier.metadata["classes"])

986.601 fluxos no teste do P2
classes do XGBoost: ['BENIGN', 'BruteForce', 'DoS']


## 1. Os vereditos e suas explicações

O teste vai à `Predictor.predict` em pedaços de 50 mil fluxos, o mesmo código que a API usa.
Ficam só os alertas, com os três atributos que os explicam.

In [2]:
rows = []
for start in range(0, len(test), 50_000):
    chunk = test.iloc[start : start + 50_000]
    results = predictor.predict(chunk)
    for position, result in enumerate(results, start=start):
        if result["verdict"] != "benign":
            features = [c["feature"] for c in result["explanation"]]
            rows.append(
                {
                    "position": position,
                    "verdict": result["verdict"],
                    "predicted": result["family"],
                    "features": features,
                }
            )
alerts = pd.DataFrame(rows)
alerts["family"] = test[schema.FAMILY].iloc[alerts["position"]].to_numpy()
alerts["explanation"] = alerts["features"].map(", ".join)
alerts["top1"] = alerts["features"].str[0]
alerts["tcp_window"] = alerts["features"].map(lambda names: bool(TCP_WINDOW & set(names)))
print(f"{len(alerts):,} alertas".replace(",", "."))

171.994 alertas


In [3]:
def most_common(values):
    value, n = Counter(values).most_common(1)[0]
    return f"{value} ({n / len(values):.0%})"


table = (
    alerts.groupby(["family", "verdict"])
    .agg(
        alertas=("verdict", "size"),
        familia_prevista=("predicted", lambda s: most_common(s.fillna("—"))),
        atributo_principal=("top1", most_common),
        janela_tcp_no_top3=("tcp_window", "mean"),
    )
    .sort_values("alertas", ascending=False)
)
table["alertas"] = table["alertas"].map(lambda n: f"{n:,}".replace(",", "."))
table["janela_tcp_no_top3"] = table["janela_tcp_no_top3"].map("{:.1%}".format)
display(table)

alertas familia_prevista  \
family       verdict                               
DDoS         attack      81.789       DoS (100%)   
BENIGN       suspicious  55.136         — (100%)   
             attack      15.817        DoS (94%)   
PortScan     suspicious   8.182         — (100%)   
DDoS         suspicious   8.179         — (100%)   
WebAttack    attack       1.999       DoS (100%)   
PortScan     attack         861        DoS (92%)   
Infiltration suspicious      26         — (100%)   
Bot          suspicious       5         — (100%)   

                                        atributo_principal janela_tcp_no_top3  
family       verdict                                                           
DDoS         attack           bwd_packet_length_mean (99%)               0.0%  
BENIGN       suspicious               psh_flag_count (55%)              66.5%  
             attack           fwd_packet_length_mean (29%)              23.5%  
PortScan     suspicious               psh_flag_count (98%)              96.8%  
DDoS         suspicious               psh_flag_count (99%)               0.0%  
WebAttack    attack                      fwd_iat_min (88%)              92.7%  
PortScan     attack      total_length_of_bwd_packets (45%)              12.2%  
Infiltration suspicious            min_packet_length (19%)              19.2%  
Bot          suspicious               flow_iat_mean (100%)               0.0%

In [4]:
# O valor mais comum da janela de volta (a de quem responde) em cada família
for family in ["WebAttack", "PortScan"]:
    window = test.loc[test[schema.FAMILY] == family, "init_win_bytes_backward"]
    share = window.value_counts(normalize=True)
    print(f"{family}: janela de volta {share.index[0]:.0f} em {share.iloc[0]:.1%} dos fluxos")

WebAttack: janela de volta 28960 em 84.8% dos fluxos
PortScan: janela de volta 0 em 98.6% dos fluxos


**Leitura.** São 171.994 alertas em 986.601 fluxos.

- O XGBoost do P2 só conhece BENIGN, BruteForce e DoS, então todo ataque que ele pega sai com uma
  dessas famílias: DDoS, ataques web e o pouco de PortScan viram DoS.
- **O DDoS é reconhecido pelo comportamento.** Em 99 % dos alertas, o atributo que mais pesa é o
  tamanho médio dos pacotes de volta, e nenhum tem um campo da janela TCP entre os três principais.
- **O ataque web, pela janela TCP.** Em 92,7 % dos alertas, um dos três campos da janela TCP está
  entre os três atributos que mais pesaram. São os atributos que o notebook 02 (seção 9) apontou
  como atalho: dependem do sistema operacional das máquinas da conversa, não do ataque (84,8 % dos
  ataques web têm a mesma janela de volta, 28960). Sem eles, a F1 dos ataques web caía de 0,96 para
  0,11 no P0.
- **No autoencoder, a janela TCP mede outra coisa.** Quase todo PortScan que ele detecta tem a
  contagem de PSH como atributo principal (98 %) e a janela TCP entre os três (96,8 %). Mas 98,6 %
  dos fluxos de varredura têm janela de volta 0, a resposta de uma porta fechada: aqui o atributo
  descreve o ataque, não a máquina.

## 2. As explicações mais comuns

Para cada tipo de alerta, os três atributos que mais se repetem como explicação, na ordem de peso.
O peso tem a escala do modelo que disparou o alerta: no XGBoost, a contribuição para o log-odds da
família prevista; no autoencoder, o erro quadrático do atributo depois da padronização. Por isso
os pesos só se comparam dentro do mesmo alerta, e a tabela mostra só os nomes.

In [5]:
def of(family, verdict):
    return (alerts["family"] == family) & (alerts["verdict"] == verdict)


groups = {
    "ataque web, visto como DoS": of("WebAttack", "attack"),
    "DDoS, visto como DoS": of("DDoS", "attack"),
    "PortScan, pelo autoencoder": of("PortScan", "suspicious"),
    "alarme falso do autoencoder": of(schema.BENIGN, "suspicious"),
}
common = {}
for title, mask in groups.items():
    explanations = alerts.loc[mask, "explanation"]
    for features, share in explanations.value_counts(normalize=True).head(2).items():
        common[(title, features)] = f"{share:.0%}"
display(pd.Series(common, name="fração dos alertas").to_frame())

fração dos alertas
ataque web, visto como DoS  fwd_iat_min, init_win_bytes_backward, fwd_header_length                                   57%
                            fwd_iat_min, init_win_bytes_backward, flow_iat_min                                        21%
DDoS, visto como DoS        bwd_packet_length_mean, bwd_packet_length_std, flow_bytes_s                               57%
                            bwd_packet_length_mean, bwd_packet_length_std, total_length_of_bwd_packets                14%
PortScan, pelo autoencoder  psh_flag_count, init_win_bytes_backward, protocol                                         62%
                            psh_flag_count, init_win_bytes_backward, flow_iat_min                                     20%
alarme falso do autoencoder psh_flag_count, init_win_bytes_backward, protocol                                         41%
                            psh_flag_count, init_win_bytes_backward, down_up_ratio                                    13%

**Leitura.** As explicações se repetem dentro de cada tipo de alerta. Os ataques web têm quase
sempre `fwd_iat_min` e `init_win_bytes_backward` à frente (78 % nas duas explicações mais comuns);
os DDoS, os tamanhos dos pacotes de volta (71 %).

E aparece um padrão estranho: a explicação mais comum dos alarmes falsos do autoencoder (41 %) é
exatamente a do PortScan que ele detecta (62 %). A próxima seção mostra o que são esses alarmes.

## 3. Os alarmes falsos do autoencoder

O notebook 03 (seção 2) mostrou que boa parte dos "normais" que disparam alarme são ataques que os
rótulos não contam: respostas da vítima ao atacante e a varredura que a máquina invadida no
Infiltration (192.168.10.8) faz na rede interna, na quinta à tarde. A mesma regra separa aqui os
alarmes falsos do autoencoder nesses dois grupos e nos normais de fato.

In [6]:
flows = pd.read_parquet(
    "../data/interim/flows.parquet",
    columns=[schema.DAY, schema.TIMESTAMP, "source_ip", "destination_ip", schema.FAMILY],
)
attacks = flows[flows[schema.FAMILY] != schema.BENIGN]
attack_pairs = set(
    zip(
        attacks[schema.DAY].astype(str),
        attacks["source_ip"],
        attacks["destination_ip"],
        strict=True,
    )
)
infiltration = flows[flows[schema.FAMILY] == "Infiltration"]
infiltration_window = infiltration.groupby("source_ip")[schema.TIMESTAMP].agg(["min", "max"])

false_alarms = alerts[of(schema.BENIGN, "suspicious")].copy()
fa = test.iloc[false_alarms["position"]]
reverse = zip(fa[schema.DAY].astype(str), fa["destination_ip"], fa["source_ip"], strict=True)
response = np.fromiter((pair in attack_pairs for pair in reverse), bool, len(fa))
scan = np.zeros(len(fa), bool)
for ip, (begin, end) in infiltration_window.iterrows():
    scan |= ((fa["source_ip"] == ip) & fa[schema.TIMESTAMP].between(begin, end)).to_numpy()
# Como no notebook 03, a resposta tem precedência sobre a varredura
false_alarms["tipo"] = np.select(
    [response, scan], ["resposta da vítima", "varredura do Infiltration"], "normal de fato"
)

by_kind = false_alarms.groupby("tipo").agg(
    alarmes=("tipo", "size"),
    atributo_principal=("top1", most_common),
    explicacao_mais_comum=("explanation", most_common),
)
display(by_kind.sort_values("alarmes", ascending=False))

pattern = alerts.loc[of("PortScan", "suspicious"), "explanation"].mode()[0]
same = false_alarms["explanation"] == pattern
print(f"explicação mais comum do PortScan pelo autoencoder: {pattern}")
print(f"alarmes falsos com essa mesma explicação: {same.sum():,}".replace(",", "."))
display(false_alarms.loc[same, "tipo"].value_counts().rename("alarmes").to_frame())

,alarmes,atributo_principal,explicacao_mais_comum
tipo,,,
varredura do Infiltration,31035,psh_flag_count (97%),"psh_flag_count, init_win_bytes_backward, protocol (73%)"
resposta da vítima,17781,urg_flag_count (96%),"urg_flag_count, down_up_ratio, fwd_iat_min (26%)"
normal de fato,6320,idle_std (17%),"idle_std, active_std, bwd_iat_std (4%)"


explicação mais comum do PortScan pelo autoencoder: psh_flag_count, init_win_bytes_backward, protocol
alarmes falsos com essa mesma explicação: 22.506


,alarmes
tipo,
varredura do Infiltration,22502
normal de fato,4


**Leitura.** A explicação separa o erro de rótulo do erro do modelo:

- Dos 22.506 alarmes "falsos" que o autoencoder explica exatamente como explica os PortScan,
  **22.502 são a varredura da máquina invadida**, rotulada como normal. Só 4 são tráfego normal de
  fato. Nesses alertas, o modelo acertou e o rótulo errou.
- As respostas da vítima (17.781 alarmes) têm outra assinatura, com a contagem de URG à frente em
  96 % deles.
- Os alarmes falsos de fato são 6.320, 11 % do total, e não têm padrão: a explicação mais comum
  cobre 4 % deles. É o que se espera de um detector de anomalias: cada fluxo normal raro é raro de
  um jeito.

## 4. Conclusões

1. **A explicação mostra, alerta por alerta, o que o notebook 02 mediu no agregado.** O XGBoost
   reconhece o DDoS pelo comportamento do fluxo e o ataque web pela janela TCP, um atalho do
   laboratório que não sobrevive a outra rede (notebook 05).
2. **No autoencoder, a explicação agrupa os alarmes.** Os que se parecem com uma varredura são uma
   varredura, mesmo quando o rótulo diz normal, e os falsos de fato não têm padrão. Numa operação de
   segurança, dá para agrupar os alertas pela explicação e revisar um grupo de cada vez, em vez de
   55 mil alertas soltos.
3. **Limites.** A explicação descreve o modelo, não o ataque: uma explicação plausível não garante
   que o alerta esteja certo, e uma explicação com a janela TCP não prova que esteja errado. E cada
   modelo tem a sua escala de peso, então a explicação de um `attack` não se compara com a de um
   `suspicious`.